In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime

In [2]:
# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_imdb_polynomial.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备Boston房价回归数据集"""
    logging.info("正在准备Boston房价回归数据集...")
    
    # 加载Boston房价数据集
    import pandas as pd
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    
    # 检查并处理数据
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    
    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1)  # 特征矩阵
    y = df['MEDV']  # 目标变量（房价中位数）
    
    # 数据标准化
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量标准化（房价通常不需要对数变换）
    y = (y - y.mean()) / y.std()  # 标准化目标变量
    
    # 划分训练测试集
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)  
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)   
   
    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
    
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    logging.info("Boston房价回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return trainloader, testloader, X_train.shape[1]

In [4]:
class PolynomialTermRegression(nn.Module):
    """回归任务专用的多项式项模块"""
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        # 回归任务不需要softplus激活，直接输出
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1))
        
    def forward(self, x):
        z = self.projection(x)
        z_powered = z.unsqueeze(-1).pow(torch.arange(self.degree + 1, device=z.device))
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CoFrNetRegression(nn.Module):
    """回归任务专用的CoFrNet"""
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([PolynomialTermRegression(input_dim, output_dim, poly_degree) for _ in range(depth)])
        self.raw_betas = nn.ParameterList([nn.Parameter(torch.full((output_dim,), 0.85)) for _ in range(depth - 1)])
        
    def forward(self, x):
        if not self.terms: 
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        # 回归任务去掉softplus激活
        output = self.terms[-1](x)
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / (output + 1e-6)  # 添加小常数避免除以零
        return output

class EnsembleResCoFrNetRegression(nn.Module):
    """回归任务专用的集成模型"""
    def __init__(self, input_dim, output_dim=1, shallow_depth=2, poly_degree=3, learning_rate=0.1):
        super().__init__()
        self.models = nn.ModuleList([])
        self.learning_rate = learning_rate
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree
        self.input_dim = input_dim
        
    def forward(self, x):
        if not self.models:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        model_outputs = [model(x) for model in self.models]
        total_prediction = sum(self.learning_rate * output for output in model_outputs)
        return total_prediction.squeeze()  # 回归任务通常输出一维
        
    def add_model(self):
        new_cofr_net = CoFrNetRegression(
            self.input_dim, 
            self.output_dim, 
            self.shallow_depth, 
            self.poly_degree
        )
        self.models.append(new_cofr_net)

In [5]:
def train_regression(ensemble_model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    best_test_loss = float('inf')
    best_model_state = None
    best_model_size = 0
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "results_per_submodel": []
    }

    logging.info("="*30)
    logging.info("  启动 Res-CoFrNet (回归版) 训练流程")
    logging.info("="*30)
    
    num_models_max = hparams['num_models_max']
    epochs_per_model = hparams['epochs_per_model']
    loss_target = hparams['loss_target']

    for i in range(num_models_max):
        submodel_log = {"submodel_index": i + 1, "training_loss_per_epoch": []}
        
        ensemble_model.add_model()
        new_model = ensemble_model.models[-1].to(device)

        for model in ensemble_model.models[:-1]:
            for param in model.parameters(): param.requires_grad = False
        for param in new_model.parameters(): param.requires_grad = True

        optimizer = optim.Adam(new_model.parameters(), 
                             lr=hparams['learning_rate_adam'], 
                             weight_decay=hparams['weight_decay'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_per_model)

        logging.info(f"--- 正在训练第 {i+1} 个子网络 ---")
        for epoch in range(epochs_per_model):
            ensemble_model.train()
            running_loss = 0.0
            for inputs, targets in trainloader:  # 注意现在是targets而不是labels
                inputs, targets = inputs.to(device), targets.float().to(device)
                optimizer.zero_grad()
                outputs = ensemble_model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            
            epoch_avg_loss = running_loss / len(trainloader)
            submodel_log["training_loss_per_epoch"].append(epoch_avg_loss)
            scheduler.step()
            logging.info(f"子网络 {i+1} | Epoch {epoch+1}/{epochs_per_model} | 训练MSE损失: {epoch_avg_loss:.4f}")

        # 验证集评估
        ensemble_model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = ensemble_model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        submodel_log["final_ensemble_loss"] = avg_test_loss
        log_data["results_per_submodel"].append(submodel_log)
        
        logging.info(f"第 {i+1} 个子网络训练完成。集成模型总体测试MSE损失: {avg_test_loss:.4f}")
        
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            best_model_state = ensemble_model.state_dict()
            best_model_size = len(ensemble_model.models)
            logging.info(f"*** 新的最佳MSE损失: {best_test_loss:.4f}，在 {i+1} 个子网络时达到。 ***")
        
        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)
        
        if best_test_loss < loss_target:
            logging.info(f"目标损失 {loss_target} 已达到。")
            break

    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型 (大小 {best_model_size}) 到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_test_loss,
            'best_model_size': best_model_size,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出任何有效模型，不进行保存。")

    return best_model_size, best_model_state

In [6]:
if __name__ == '__main__':
    # --- 文件与超参数定义 ---
    LOG_FILE = 'training_regression.log'
    JSON_LOG_FILE = 'training_regression_log.json'
    MODEL_SAVE_PATH = 'regression_res_cofrnet_polynomial.pth'
    
    HYPERPARAMETERS = {
        "num_models_max": 5,
        "loss_target": 0.1,  # MSE目标值
        "epochs_per_model": 200,
        "shallow_depth_per_cofrnet": 2,
        "polynomial_degree": 3,
        "boosting_learning_rate": 0.1,
        "learning_rate_adam": 1e-3,  # 回归任务通常需要更大的学习率
        "weight_decay": 1e-5,
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()
    
    # 创建回归版的集成网络
    res_cofr_net_reg = EnsembleResCoFrNetRegression(
        input_dim=input_dim, 
        output_dim=1,  # 回归任务输出维度为1
        shallow_depth=HYPERPARAMETERS['shallow_depth_per_cofrnet'],
        poly_degree=HYPERPARAMETERS['polynomial_degree'], 
        learning_rate=HYPERPARAMETERS['boosting_learning_rate']
    ).to(device)

    start_time = time.time()
    best_size, best_state_dict = train_regression(
        res_cofr_net_reg, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    
    # 加载并验证保存的最佳模型
    if best_state_dict:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳回归模型")
        logging.info("="*30)
        
        verification_model = EnsembleResCoFrNetRegression(
            input_dim=input_dim, output_dim=1,
            shallow_depth=HYPERPARAMETERS['shallow_depth_per_cofrnet'],
            poly_degree=HYPERPARAMETERS['polynomial_degree'], 
            learning_rate=HYPERPARAMETERS['boosting_learning_rate']
        )
        for _ in range(best_size):
            verification_model.add_model()

        checkpoint = torch.load(MODEL_SAVE_PATH)
        verification_model.load_state_dict(checkpoint['model_state_dict'])
        verification_model.to(device)
        verification_model.eval()

        logging.info(f"成功加载模型。模型大小: {checkpoint['best_model_size']} 个子网络, 保存的MSE损失: {checkpoint['best_loss']:.4f}")

        total_params = sum(p.numel() for p in verification_model.parameters() if p.requires_grad)
        logging.info(f"模型总参数数量: {total_params}")

        test_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")
        
        assert abs(avg_test_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
        logging.info("验证成功！回归模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存回归模型。")

2025-08-21 12:13:34,511 [INFO ]  回归实验超参数: {
    "num_models_max": 5,
    "loss_target": 0.1,
    "epochs_per_model": 200,
    "shallow_depth_per_cofrnet": 2,
    "polynomial_degree": 3,
    "boosting_learning_rate": 0.1,
    "learning_rate_adam": 0.001,
    "weight_decay": 1e-05
}
2025-08-21 12:13:34,512 [INFO ]  正在准备Boston房价回归数据集...
2025-08-21 12:13:34,688 [INFO ]  NumExpr defaulting to 4 threads.
2025-08-21 12:13:35,739 [INFO ]  Boston房价回归数据集准备完毕。
2025-08-21 12:13:35,740 [INFO ]  训练样本数: 354, 测试样本数: 152
2025-08-21 12:13:35,741 [INFO ]  特征维度: 13, 目标维度: 1
2025-08-21 12:13:35,742 [INFO ]  ==============================
2025-08-21 12:13:35,744 [INFO ]    启动 Res-CoFrNet (回归版) 训练流程
2025-08-21 12:13:35,745 [INFO ]  ==============================
2025-08-21 12:13:39,307 [INFO ]  --- 正在训练第 1 个子网络 ---
2025-08-21 12:13:39,386 [INFO ]  子网络 1 | Epoch 1/200 | 训练MSE损失: 4562.6698
2025-08-21 12:13:39,397 [INFO ]  子网络 1 | Epoch 2/200 | 训练MSE损失: 1.7466
2025-08-21 12:13:39,408 [INFO ]  子网络 1 | Epoch 3/20